# 5-Hop Reasoning Pipeline with Flan-T5-XXL

Test **LLMClient** with `provider="flan_t5"` (google/flan-t5-xxl) and the **5-hop ReasoningPipeline**.

- **Backend**: Hugging Face Transformers (not Ollama). Model loads on first `generate()`.
- **Requirements**: `pip install transformers torch` (optional GPU: `accelerate`).
- **Usage**: Run cells in order. First inference may take a minute while the model loads.

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path().resolve().parent))

# Optional: install Flan-T5 dependencies (uncomment on Colab if needed)
# !pip install -q transformers torch
# !pip install -q accelerate  # for GPU

## 2. Create LLMClient (Flan-T5-XXL) and Pipeline

In [ ]:
from src.pipeline import ReasoningPipeline
from src.pipeline.llm_client import LLMClient

llm = LLMClient(
    provider="flan_t5",
    model="google/flan-t5-xxl",
    max_tokens=256,
    temperature=0.0,
    device=None,  # auto: cuda if available, else cpu
)

pipeline = ReasoningPipeline(llm_client=llm)
print(f"Provider: {llm.provider}")
print("Model: google/flan-t5-xxl (loads on first generate)")

## 3. Run 5-Hop Pipeline on Sample Headlines

In [ ]:
samples = [
    {
        "headline": "Euro to benefit from the ECB's pronounced hawkish determination – Commerzbank",
        "ticker": "EURCHF",
    },
    {
        "headline": "EURCHF could extend its advance back to levels between 1.02 and 1.04 – MUFG",
        "ticker": "EURCHF",
    },
    {
        "headline": "New lows for the EURUSD. EURCHF down as well and tests its 200 day MA.",
        "ticker": "EURCHF",
    },
]

for i, s in enumerate(samples):
    print("=" * 60)
    print(f"Sample {i + 1}: {s['headline'][:70]}...")
    print(f"Ticker: {s['ticker']}")
    print("-" * 60)
    try:
        context = pipeline.run(s["headline"], ticker=s["ticker"])
        print(f"Hop 1 – Entity:        {context.primary_entity}")
        print(f"Hop 2 – Aspect:        {context.primary_aspect}")
        print(f"Hop 3 – Implicit cues: {context.implicit_cues or 'None'}")
        print(f"Hop 4 – Sentiment:     {context.sentiment}")
        print(f"Hop 5 – Market:        {context.market_implication}")
    except Exception as e:
        print(f"Error: {e}")
    print()

## 4. Single Run with Full Context (Pretty Print)

In [ ]:
headline = "EURCHF Room for the Euro to extend the move higher – MUFG"
ticker = "EURCHF"

context = pipeline.run(headline, ticker=ticker)

print("Headline:", context.text)
print("Ticker:", context.ticker)
print()
print("Hop 1 – Entity grounding:", context.primary_entity)
print("Hop 2 – Financial aspect:", context.primary_aspect)
print("Hop 3 – Implicit cues:", context.implicit_cues)
print("Hop 4 – Sentiment:", context.sentiment)
print("Hop 5 – Market implication:", context.market_implication)
print()
print("Usage:", llm.get_usage_stats())

## 5. Optional: Run on Dataset Samples

If `sentiment_annotated_with_texts.csv` (or single-article CSV) is available, run the pipeline on a few rows.

In [ ]:
import pandas as pd
from src.utils.data_loader import load_all_dataframes

base = Path().resolve().parent
if (base / "sentiment_annotated_with_texts.csv").exists() or (
    base / "sentiment_predictions_single_article.csv"
).exists():
    data = load_all_dataframes(base)
    df = data.get("single_article") or data.get("ground_truth")
    if df is not None and len(df) > 0:
        # Use title or text column
        text_col = "title" if "title" in df.columns else "text"
        ticker_col = "ticker" if "ticker" in df.columns else None
        row = df.iloc[0]
        text = row.get(text_col) or row.get("text") or ""
        ticker = (
            str(row[ticker_col]).strip()
            if ticker_col and pd.notna(row.get(ticker_col))
            else None
        )
        if text:
            print("From dataset:", text[:80], "...")
            ctx = pipeline.run(text, ticker=ticker)
            print("Sentiment:", ctx.sentiment, "| Market:", ctx.market_implication)
    else:
        print("No single_article or ground_truth dataframe found.")
else:
    print("Dataset CSV not found; skip or add path.")